# OYKHCHAR LoRA Training - Fixed Version

**Fixed Issues:**
- [OK] Added Hugging Face authentication
- [OK] Memory-optimized for Kaggle free tier
- [OK] Using proven ai-toolkit training script
- [OK] Proper FLUX LoRA implementation

**Setup Required:**
1. Add Hugging Face token to Kaggle Secrets:
   - Go to: Account  Settings  Secrets
   - Add new secret: `HF_TOKEN` = your_token
   - Get token from: https://huggingface.co/settings/tokens

**Training Time:** ~45-60 minutes on T4 GPU

In [ ]:
# Install ai-toolkit (proven LoRA training framework)
!git clone https://github.com/ostris/ai-toolkit.git
%cd ai-toolkit

# Install dependencies
!pip install -q torch torchvision
!pip install -q -r requirements.txt

print("[OK] ai-toolkit installed")

In [ ]:
# Setup Hugging Face authentication
from kaggle_secrets import UserSecretsClient
import os

# Get HF token from Kaggle secrets
user_secrets = UserSecretsClient()
try:
    hf_token = user_secrets.get_secret("HF_TOKEN")
    os.environ['HF_TOKEN'] = hf_token
    print("[OK] Hugging Face token loaded from secrets")
except:
    print("[ERROR] HF_TOKEN not found in secrets!")
    print("")
    print("Please add your Hugging Face token:")
    print("1. Go to https://huggingface.co/settings/tokens")
    print("2. Create a token (read permission needed)")
    print("3. Go to Kaggle Account  Settings  Secrets")
    print("4. Add secret: HF_TOKEN = your_token_here")
    print("5. Re-run this notebook")
    raise Exception("HF_TOKEN required")

# Login to Hugging Face
!huggingface-cli login --token $HF_TOKEN
print("[OK] Logged into Hugging Face")

In [ ]:
# Extract training dataset
import zipfile
from pathlib import Path

# Find the dataset
dataset_path = '/kaggle/input/oykhchar-lora-images/images.zip'
extract_path = '/kaggle/working/training_data'

print(f"Extracting dataset from {dataset_path}...")
os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(dataset_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

# Count files
image_files = list(Path(extract_path).rglob('*.jpg')) + list(Path(extract_path).rglob('*.png'))
caption_files = list(Path(extract_path).rglob('*.txt'))

print(f"[OK] Extracted {len(image_files)} images")
print(f"[OK] Found {len(caption_files)} captions")
print(f"[FOLDER] Dataset location: {extract_path}")

# Show sample
if image_files:
    print(f"\nSample files:")
    for img in image_files[:3]:
        print(f"  {img.name}")
        caption_file = img.with_suffix('.txt')
        if caption_file.exists():
            caption = caption_file.read_text().strip()
            print(f"     {caption[:60]}...")

In [ ]:
# Create training configuration for ai-toolkit
import yaml

# Find the actual image directory (might be nested)
image_dir = None
for path in Path(extract_path).rglob('*.jpg'):
    image_dir = str(path.parent)
    break

if not image_dir:
    for path in Path(extract_path).rglob('*.png'):
        image_dir = str(path.parent)
        break

print(f"Images directory: {image_dir}")

config = {
    'job': 'extension',
    'config': {
        'name': 'oykhchar_lora',
        'process': [
            {
                'type': 'sd_trainer',
                'training_folder': '/kaggle/working/output',
                'device': 'cuda:0',
                'network': {
                    'type': 'lora',
                    'linear': 16,
                    'linear_alpha': 16
                },
                'save': {
                    'dtype': 'float16',
                    'save_every': 250,
                    'max_step_saves_to_keep': 2
                },
                'datasets': [
                    {
                        'folder_path': image_dir,
                        'caption_ext': 'txt',
                        'caption_dropout_rate': 0.05,
                        'shuffle_tokens': False,
                        'cache_latents_to_disk': True,
                        'resolution': [512, 768, 1024]
                    }
                ],
                'train': {
                    'batch_size': 1,
                    'steps': 1000,
                    'gradient_accumulation_steps': 1,
                    'train_unet': True,
                    'train_text_encoder': False,
                    'gradient_checkpointing': True,
                    'noise_scheduler': 'flowmatch',
                    'optimizer': 'adamw8bit',
                    'lr': 4e-4,
                    'skip_first_sample': True,
                    'linear_timesteps': True,
                    'ema_config': {
                        'use_ema': True,
                        'ema_decay': 0.99
                    },
                    'dtype': 'bf16'
                },
                'model': {
                    'name_or_path': 'black-forest-labs/FLUX.1-dev',
                    'is_flux': True,
                    'quantize': True
                },
                'sample': {
                    'sampler': 'flowmatch',
                    'sample_every': 250,
                    'width': 1024,
                    'height': 1024,
                    'prompts': [
                        'OYKHCHAR character standing with arms raised',
                        'OYKHCHAR character sitting and thinking',
                        'OYKHCHAR character pointing forward'
                    ],
                    'neg': '',
                    'seed': 42,
                    'walk_seed': True,
                    'guidance_scale': 3.5,
                    'sample_steps': 28
                }
            }
        ]
    }
}

# Save config
config_path = '/kaggle/working/training_config.yaml'
with open(config_path, 'w') as f:
    yaml.dump(config, f, default_flow_style=False)

print(f"[OK] Training config created: {config_path}")
print("\nConfig preview:")
print(yaml.dump(config, default_flow_style=False)[:500] + "...")

In [ ]:
# Check GPU and memory
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    
    # Clear cache
    torch.cuda.empty_cache()
    print("[OK] GPU ready for training")
else:
    print("[ERROR] No GPU available! Enable GPU in notebook settings.")

In [ ]:
# Start training
print("[TRAINING] Starting LoRA training...")
print("="*60)
print("This will take approximately 45-60 minutes")
print("You can monitor progress below")
print("="*60)
print()

!python run.py /kaggle/working/training_config.yaml

In [ ]:
# Check training outputs
import os

output_dir = '/kaggle/working/output'
print("Training outputs:")
print()

if os.path.exists(output_dir):
    !ls -lh {output_dir}
    
    # Find LoRA files
    lora_files = list(Path(output_dir).rglob('*.safetensors'))
    if lora_files:
        print(f"\n[OK] Found {len(lora_files)} LoRA checkpoint(s):")
        for lora in lora_files:
            size_mb = lora.stat().st_size / 1024 / 1024
            print(f"  [PACKAGE] {lora.name} ({size_mb:.1f} MB)")
    else:
        print("\n[WARNING] No .safetensors files found yet")
    
    # Show sample images
    sample_imgs = list(Path(output_dir).rglob('*.png')) + list(Path(output_dir).rglob('*.jpg'))
    if sample_imgs:
        print(f"\n[OK] Found {len(sample_imgs)} sample image(s)")
        
        # Display first few samples
        from PIL import Image
        from IPython.display import display
        
        for img_path in sample_imgs[:3]:
            print(f"\n{img_path.name}:")
            img = Image.open(img_path)
            display(img.resize((512, 512)))
else:
    print(f"[ERROR] Output directory not found: {output_dir}")

In [ ]:
# Package final LoRA for download
import shutil

output_dir = '/kaggle/working/output'
final_dir = '/kaggle/working/oykhchar_lora_final'

os.makedirs(final_dir, exist_ok=True)

# Copy all .safetensors files
lora_files = list(Path(output_dir).rglob('*.safetensors'))
for lora_file in lora_files:
    shutil.copy(lora_file, final_dir)
    print(f"[OK] Copied: {lora_file.name}")

# Copy sample images
sample_imgs = list(Path(output_dir).rglob('*.png')) + list(Path(output_dir).rglob('*.jpg'))
for img in sample_imgs:
    shutil.copy(img, final_dir)
    
print(f"\n[PACKAGE] Final LoRA package ready: {final_dir}")
print("\nTo download:")
print("1. Click 'Output' tab in Kaggle")
print("2. Download the oykhchar_lora_final folder")
print("3. Use the .safetensors file with FLUX")

!ls -lh {final_dir}

## [OK] Training Complete!

### Next Steps:

1. **Download the LoRA weights** from the Output tab
2. **Test the LoRA** with your FLUX pipeline
3. **Use trigger word:** `OYKHCHAR` in all prompts

### Example Usage:

```python
from diffusers import FluxPipeline

pipe = FluxPipeline.from_pretrained(
    "black-forest-labs/FLUX.1-dev",
    torch_dtype=torch.bfloat16
)
pipe.load_lora_weights("path/to/oykhchar_lora_final")

image = pipe(
    "OYKHCHAR character pointing at viewer",
    num_inference_steps=28,
    guidance_scale=3.5
).images[0]
```